# Home Credit Default Risk: Borrower Segmentation via Contrastive Representation Learning

**Authors:** Wade Chen, Manuel Antonio Arce Aguirre, Rita Luo, Qiqi Huang, Nick Dhaliwal, Yingqi Chen

## 1. Introduction & Problem Statement

Credit risk modeling has traditionally focused on supervised prediction (e.g., default vs non-default). While predictive accuracy is important, lenders also need **interpretable borrower segments** to support strategy design, product personalization, and risk-aware portfolio management.

Classical unsupervised methods on high-dimensional tabular credit data often produce weakly separated clusters after linear reduction, limiting downstream interpretability. This project investigates whether modern self-supervised tabular representation learning can produce embeddings with stronger geometric structure.

### Motivation
- Borrower portfolios are heterogeneous and potentially multi-modal.
- Linear embeddings (e.g., PCA) may not capture non-linear borrower relationships.
- Better latent spaces can improve unsupervised segmentation and policy insights.

### Literature Context
- **SCARF (ICLR 2022):** Self-supervised contrastive learning framework for tabular data using feature corruption to generate positive views.
- **ReConTab (arXiv 2023):** Contrastive representation learning for tabular data emphasizing robust latent structure under realistic corruption/augmentation.
- **TCCL (arXiv 2025):** Recent contrastive tabular learning direction that further improves cluster-friendly embeddings and representation transfer.

### Project Arc
1. Build strong preprocessing + EDA foundation on `application_train.csv`.
2. Run baseline unsupervised pipeline: **PCA + K-Means / Hierarchical / DBSCAN**.
3. Implement **SCARF from scratch in PyTorch** and learn 64-d embeddings.
4. Re-run clustering in SCARF space and compare structure, silhouette, and risk separation.

The narrative target is explicit: baseline latent geometry is comparatively overlapping, while contrastive embeddings reveal more distinct risk islands.


In [ ]:
# Core
import os
import random
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm

# Scikit-learn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score, roc_auc_score
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

# SciPy / UMAP
from scipy.cluster.hierarchy import linkage, dendrogram
import umap.umap_ as umap

# Optional knee detector for DBSCAN eps tuning
try:
    from kneed import KneeLocator
    KNEED_AVAILABLE = True
except Exception:
    KneeLocator = None
    KNEED_AVAILABLE = False

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10

# Reproducible RNG helper
rng = np.random.default_rng(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"KneeLocator available: {KNEED_AVAILABLE}")


## 2. Data Loading & Preprocessing

We load the Home Credit training table and **split immediately** to avoid leakage:
- Separate `TARGET` and `SK_ID_CURR`
- Create an 80/20 stratified split right away
- Fit preprocessing steps on **train only** and apply to test:
  - missing indicators for columns with >5% train missingness
  - median imputation for numeric features
  - mode imputation for categorical features
  - ordinal encode `NAME_EDUCATION_TYPE`
  - one-hot encode nominal categoricals on train, then align test columns via `reindex`
  - standardize with `StandardScaler` fit on train only
  - PCA fit on train only

This follows the proper split-first pipeline pattern and prevents test-set statistics from leaking into training.


In [ ]:
DATA_CANDIDATES = [
    "application_train.csv",
    "/kaggle/input/home-credit-default-risk/application_train.csv",
]

data_path = None
for candidate in DATA_CANDIDATES:
    if os.path.exists(candidate):
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError(
        "application_train.csv not found. Place it in the working directory "
        "or use Kaggle path /kaggle/input/home-credit-default-risk/application_train.csv"
    )

df_raw = pd.read_csv(data_path)
df = df_raw.copy()

if "TARGET" not in df.columns:
    raise ValueError("TARGET column is required in application_train.csv")

target_series = df["TARGET"].astype(int).copy()
id_series = df["SK_ID_CURR"].copy() if "SK_ID_CURR" in df.columns else pd.Series(df.index, index=df.index, name="SK_ID_CURR")

drop_cols = ["TARGET"]
if "SK_ID_CURR" in df.columns:
    drop_cols.append("SK_ID_CURR")

X_raw_df = df.drop(columns=drop_cols).copy()

# Split immediately after separating TARGET and ID to prevent leakage.
X_train_raw_df, X_test_raw_df, target_train, target_test = train_test_split(
    X_raw_df,
    target_series,
    test_size=0.2,
    random_state=SEED,
    stratify=target_series,
)

target_train = target_train.astype(int).copy()
target_test = target_test.astype(int).copy()
id_train = id_series.loc[X_train_raw_df.index].copy()
id_test = id_series.loc[X_test_raw_df.index].copy()

print(f"Loaded dataset from: {data_path}")
print(f"Raw feature matrix shape (before preprocessing): {X_raw_df.shape}")
print(f"Train rows: {X_train_raw_df.shape[0]} | Test rows: {X_test_raw_df.shape[0]}")
print(f"Train default rate: {target_train.mean():.4f} | Test default rate: {target_test.mean():.4f}")


In [ ]:
print("Shape:", df.shape)
print("\nDtype counts:")
display(df.dtypes.value_counts().rename("count").to_frame())
print("\nHead:")
display(df.head())
print("\nInfo:")
df.info()


In [ ]:
# Train-only preprocessing to eliminate leakage

X_train_proc = X_train_raw_df.copy()
X_test_proc = X_test_raw_df.copy()

# Add missingness indicators based on TRAIN missing rates only.
missing_ratio_train = X_train_proc.isnull().mean()
high_missing_cols = missing_ratio_train[missing_ratio_train > 0.05].index.tolist()
for col in high_missing_cols:
    indicator_col = f"{col}_MISSING"
    X_train_proc[indicator_col] = X_train_proc[col].isnull().astype(np.int8)
    X_test_proc[indicator_col] = X_test_proc[col].isnull().astype(np.int8)

numeric_cols = X_train_proc.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train_proc.select_dtypes(exclude=[np.number]).columns.tolist()

# Keep label encoding only for truly ordinal categories.
ordinal_cols = [c for c in ["NAME_EDUCATION_TYPE"] if c in categorical_cols]
nominal_cols = [c for c in categorical_cols if c not in ordinal_cols]

# Fix DAYS_EMPLOYED anomaly BEFORE imputation: 365243 is a placeholder for unknown.
if "DAYS_EMPLOYED" in X_train_proc.columns:
    X_train_proc.loc[X_train_proc["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    X_test_proc.loc[X_test_proc["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

# Impute numeric with TRAIN medians
numeric_impute_values = {}
for col in tqdm(numeric_cols, desc="Imputing numeric columns (train medians)"):
    median_val = X_train_proc[col].median()
    numeric_impute_values[col] = median_val
    X_train_proc[col] = X_train_proc[col].fillna(median_val)
    X_test_proc[col] = X_test_proc[col].fillna(median_val)

# Impute categorical with TRAIN modes
categorical_impute_values = {}
for col in tqdm(categorical_cols, desc="Imputing categorical columns (train modes)"):
    mode_vals = X_train_proc[col].mode(dropna=True)
    fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else "Unknown"
    categorical_impute_values[col] = fill_val
    X_train_proc[col] = X_train_proc[col].fillna(fill_val)
    X_test_proc[col] = X_test_proc[col].fillna(fill_val)

# Label encoding for ordinal education only (ordered mapping via numeric prefixes).
label_encoders = {}
if "NAME_EDUCATION_TYPE" in ordinal_cols:
    education_map = {
        "Lower secondary": "0_Lower secondary",
        "Secondary / secondary special": "1_Secondary / secondary special",
        "Incomplete higher": "2_Incomplete higher",
        "Higher education": "3_Higher education",
        "Academic degree": "4_Academic degree",
    }
    education_order = [
        "0_Lower secondary",
        "1_Secondary / secondary special",
        "2_Incomplete higher",
        "3_Higher education",
        "4_Academic degree",
        "5_Unknown",
    ]

    edu_col = "NAME_EDUCATION_TYPE"
    X_train_proc[edu_col] = X_train_proc[edu_col].map(education_map).fillna("5_Unknown")
    X_test_proc[edu_col] = X_test_proc[edu_col].map(education_map).fillna("5_Unknown")

    le = LabelEncoder()
    le.fit(education_order)
    X_train_proc[edu_col] = le.transform(X_train_proc[edu_col].astype(str))
    X_test_proc[edu_col] = le.transform(X_test_proc[edu_col].astype(str))
    label_encoders[edu_col] = le

# Keep copies for interpretation/profiling before one-hot expansion.
X_train_profile_df = X_train_proc.copy()
X_test_profile_df = X_test_proc.copy()

# One-hot encode nominal categoricals using train-fitted categories for exact alignment.
pre_dummy_columns = set(X_train_proc.columns)
if len(nominal_cols) > 0:
    # Use pd.Categorical to force test to use train's categories before dummying
    for col in nominal_cols:
        train_categories = X_train_proc[col].unique().tolist()
        X_train_proc[col] = pd.Categorical(X_train_proc[col], categories=sorted(train_categories))
        X_test_proc[col] = pd.Categorical(X_test_proc[col], categories=sorted(train_categories))

    X_train_model_df = pd.get_dummies(X_train_proc, columns=nominal_cols, drop_first=True, dtype=np.uint8)
    X_test_model_df = pd.get_dummies(X_test_proc, columns=nominal_cols, drop_first=True, dtype=np.uint8)
    # Align columns (handles any truly unseen categories gracefully)
    X_test_model_df = X_test_model_df.reindex(columns=X_train_model_df.columns, fill_value=0)
else:
    X_train_model_df = X_train_proc.copy()
    X_test_model_df = X_test_proc.copy()

feature_columns = X_train_model_df.columns.tolist()

# Track one-hot groups for group-level categorical corruption in SCARF.
feature_to_idx = {col: i for i, col in enumerate(feature_columns)}
cat_groups = {}
for base_col in nominal_cols:
    prefix = f"{base_col}_"
    group_cols = [c for c in feature_columns if c.startswith(prefix) and c not in pre_dummy_columns]
    if len(group_cols) > 0:
        cat_groups[base_col] = [feature_to_idx[c] for c in group_cols]
cat_group_indices = list(cat_groups.values())

# Bundle _MISSING indicators with their parent features for joint SCARF corruption.
for indicator_col in [f"{c}_MISSING" for c in high_missing_cols]:
    if indicator_col not in feature_to_idx:
        continue
    indicator_idx = feature_to_idx[indicator_col]
    parent_col = indicator_col.replace("_MISSING", "")

    # Check if parent is in an existing categorical group
    bundled = False
    for group_name, group_indices in cat_groups.items():
        if parent_col == group_name or indicator_col.startswith(group_name + "_"):
            group_indices.append(indicator_idx)
            bundled = True
            break

    if not bundled and parent_col in feature_to_idx:
        # Parent is a numeric feature - create a new group [parent, indicator]
        parent_idx = feature_to_idx[parent_col]
        cat_groups[f"{parent_col}_with_missing"] = [parent_idx, indicator_idx]

# Rebuild cat_group_indices and remove grouped columns from numeric_indices in ScarfDataset
cat_group_indices = list(cat_groups.values())

# Standardize on TRAIN only.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_model_df).astype(np.float32)
X_test_scaled = scaler.transform(X_test_model_df).astype(np.float32)

# Combined matrices only for EDA/display (no fitting uses these).
processed_train_df = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X_train_model_df.index)
processed_test_df = pd.DataFrame(X_test_scaled, columns=feature_columns, index=X_test_model_df.index)
processed_df = pd.concat([processed_train_df, processed_test_df], axis=0).sort_index()
X_scaled = processed_df.to_numpy(dtype=np.float32)

df_proc = pd.concat(
    [
        X_train_model_df.assign(TARGET=target_train),
        X_test_model_df.assign(TARGET=target_test),
    ],
    axis=0,
).sort_index()

print("Processed TRAIN feature matrix shape:", X_train_model_df.shape)
print("Processed TEST feature matrix shape:", X_test_model_df.shape)
print("Scaled TRAIN shape:", X_train_scaled.shape)
print("Scaled TEST shape:", X_test_scaled.shape)
print("Residual missing values after transform (train/test):", int(np.isnan(X_train_scaled).sum()), int(np.isnan(X_test_scaled).sum()))
print("Missingness indicators added from train (>5% missing):", len(high_missing_cols))
print("Nominal categorical columns one-hot encoded:", len(nominal_cols))
print("Ordinal columns label encoded:", ordinal_cols if len(ordinal_cols) > 0 else "None")
print("Categorical one-hot groups tracked for SCARF corruption:", len(cat_group_indices))


In [ ]:
print("Scaled feature summary (first 15 rows):")
display(processed_df.describe().T.head(15))

print("\nTrain target distribution:")
display(target_train.value_counts().rename("count").to_frame())
display((target_train.value_counts(normalize=True) * 100).rename("percent").to_frame())

print("\nTest target distribution:")
display(target_test.value_counts().rename("count").to_frame())
display((target_test.value_counts(normalize=True) * 100).rename("percent").to_frame())


## 3. Exploratory Data Analysis (EDA)

We characterize class imbalance, feature distributions, correlation structure, and pre-imputation missingness patterns. This provides context for why representation learning may be necessary.


In [ ]:
# TARGET class distribution

target_counts = target_series.value_counts().sort_index()
target_pct = (target_counts / target_counts.sum()) * 100

fig, ax = plt.subplots(figsize=(8, 5))
bar = ax.bar(target_counts.index.astype(str), target_counts.values, color=sns.color_palette("Set2", 2))
ax.set_title("TARGET Class Distribution")
ax.set_xlabel("TARGET")
ax.set_ylabel("Count")

for i, rect in enumerate(bar):
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        rect.get_height(),
        f"{target_pct.iloc[i]:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.legend(["Class 0/1 counts"], loc="upper right")
plt.show()


In [ ]:
# Distribution of key numeric features

key_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
]

available_key_features = [c for c in key_features if c in df_proc.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(available_key_features):
    sns.histplot(df_proc[feat], bins=50, kde=True, ax=axes[i], color=sns.color_palette("Set2", 8)[i % 8])
    axes[i].set_title(f"Distribution: {feat}")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Frequency")

# Hide unused subplot if any
for j in range(len(available_key_features), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap: top 20 features most correlated with TARGET

corr_with_target = df_proc.corr(numeric_only=True)["TARGET"].drop("TARGET")
top20 = corr_with_target.abs().sort_values(ascending=False).head(20).index.tolist()

corr_mat = df_proc[top20 + ["TARGET"]].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_mat, cmap="coolwarm", center=0, annot=False, linewidths=0.5)
plt.title("Correlation Heatmap: Top 20 Features Correlated with TARGET")
plt.xlabel("Features")
plt.ylabel("Features")
plt.show()

print("Top 20 absolute correlations with TARGET:")
display(corr_with_target.loc[top20].sort_values(key=np.abs, ascending=False).rename("corr_with_target").to_frame())


In [ ]:
# Box plots of key features by TARGET class

box_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
]
box_features = [c for c in box_features if c in df_proc.columns]

# Sample for plotting speed
box_n = min(50000, len(df_proc))
box_idx = rng.choice(len(df_proc), size=box_n, replace=False)
box_df = df_proc.iloc[box_idx].copy()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(box_features):
    sns.boxplot(data=box_df, x="TARGET", y=feat, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{feat} by TARGET")
    axes[i].set_xlabel("TARGET")
    axes[i].set_ylabel(feat)

for j in range(len(box_features), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Missingness visualization based on raw (pre-imputation) data

missing_pct = (df_raw.isnull().mean() * 100).sort_values(ascending=False)
missing_pct_nonzero = missing_pct[missing_pct > 0]

plt.figure(figsize=(12, 6))
missing_pct_nonzero.head(20).sort_values().plot(kind="barh", color=sns.color_palette("Set2", 8)[2])
plt.title("Top 20 Features by Missing Value Percentage (Raw Data)")
plt.xlabel("Missing Percentage (%)")
plt.ylabel("Feature")
plt.show()

# Missingness heatmap for top missing columns (sampled rows for readability)
top_missing_cols = missing_pct_nonzero.head(20).index.tolist()
if len(top_missing_cols) > 0:
    sample_n = min(2000, len(df_raw))
    heatmap_df = df_raw[top_missing_cols].sample(n=sample_n, random_state=SEED).isnull().astype(int)

    plt.figure(figsize=(14, 6))
    sns.heatmap(heatmap_df.T, cmap="mako", cbar=True)
    plt.title("Missingness Heatmap (Top Missing Columns, Sampled Rows)")
    plt.xlabel("Sampled Row Index")
    plt.ylabel("Feature")
    plt.show()
else:
    print("No missing values found in raw data.")


## 4. Baseline - PCA + Classical Clustering

We build a classical unsupervised baseline by reducing dimension with PCA, then applying K-Means, hierarchical clustering (Ward linkage), and DBSCAN.

Leakage-safe protocol:
- PCA is fit on **train only**, then applied to test
- K-Means is fit on train and evaluated on test
- DBSCAN is run on a test sample with `eps` tuned from each space's k-distance curve

For DBSCAN, `eps` is selected using **maximum curvature** from `kneed.KneeLocator` when available, with a secant-line fallback if `kneed` is unavailable.
Using the same tuning logic in PCA and SCARF spaces makes the density-clustering comparison fair.


In [ ]:
def silhouette_on_sample(X, labels, max_points=50000, seed=42):
    # Compute silhouette robustly on a sample for scalability.
    unique = np.unique(labels)
    if len(unique) < 2:
        return np.nan

    n = X.shape[0]
    if n > max_points:
        local_rng = np.random.default_rng(seed)
        idx = local_rng.choice(n, size=max_points, replace=False)
        return silhouette_score(X[idx], labels[idx])
    return silhouette_score(X, labels)


def k_distance_elbow_eps(X, min_samples=12):
    # DBSCAN k-distance elbow with KneeLocator maximum-curvature fallback.
    nn_model = NearestNeighbors(n_neighbors=min_samples, n_jobs=-1)
    nn_model.fit(X)
    distances, _ = nn_model.kneighbors(X)

    # Descending order to match classic visual elbow interpretation.
    kth_dist = np.sort(distances[:, -1])[::-1]
    x = np.arange(1, len(kth_dist) + 1)

    elbow_idx = None
    method = "secant_fallback"

    if KNEED_AVAILABLE:
        knee = KneeLocator(x, kth_dist, curve="convex", direction="decreasing")
        if knee.knee is not None:
            elbow_idx = int(knee.knee - 1)
            method = "kneed_max_curvature"

    if elbow_idx is None:
        baseline_line = np.linspace(kth_dist[0], kth_dist[-1], len(kth_dist))
        deviation = np.abs(kth_dist - baseline_line)
        elbow_idx = int(np.argmax(deviation))

    eps = float(kth_dist[elbow_idx])
    return kth_dist, elbow_idx, eps, method


def silhouette_bootstrap_scores(X, labels, n_boot=10, sample_size=50000, seed=42):
    # Repeated subsampling to estimate silhouette uncertainty.
    scores = []
    n = X.shape[0]
    m = min(sample_size, n)

    for i in range(n_boot):
        local_rng = np.random.default_rng(seed + i)
        idx = local_rng.choice(n, size=m, replace=False)

        local_labels = labels[idx]
        if len(np.unique(local_labels)) < 2:
            continue

        scores.append(silhouette_score(X[idx], local_labels))

    return np.array(scores, dtype=float)


# PCA fit on TRAIN only for explained variance
pca_full = PCA(random_state=SEED)
pca_full.fit(X_train_scaled)

explained_var = pca_full.explained_variance_ratio_
cum_explained = np.cumsum(explained_var)
n_components_90 = int(np.argmax(cum_explained >= 0.90) + 1)

print(f"Components needed for ~90% variance (train-fit): {n_components_90}")

# Final PCA representation: fit on train, transform train/test.
pca = PCA(n_components=n_components_90, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

max_scree = min(60, len(explained_var))
axes[0].plot(np.arange(1, max_scree + 1), explained_var[:max_scree], marker="o")
axes[0].set_title("PCA Scree Plot (First 60 Components, Train Fit)")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")

axes[1].plot(np.arange(1, len(cum_explained) + 1), cum_explained, marker=".")
axes[1].axhline(0.90, color="red", linestyle="--", label="90% variance")
axes[1].axvline(n_components_90, color="green", linestyle="--", label=f"n={n_components_90}")
axes[1].set_title("Cumulative Explained Variance (Train Fit)")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].legend()

plt.tight_layout()
plt.show()

print("PCA train shape:", X_train_pca.shape)
print("PCA test shape:", X_test_pca.shape)


In [ ]:
# K-Means model selection on PCA TRAIN features

k_values = list(range(2, 11))
inertias_pca = []
silhouette_scores_pca = []

for k in tqdm(k_values, desc="K-Means on PCA train (k=2..10)"):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED)
    labels_train = km.fit_predict(X_train_pca)

    inertias_pca.append(km.inertia_)
    sil = silhouette_on_sample(X_train_pca, labels_train, max_points=50000, seed=SEED)
    silhouette_scores_pca.append(sil)

best_k_pca = k_values[int(np.nanargmax(silhouette_scores_pca))]
print(f"Best k by silhouette (PCA train): {best_k_pca}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(k_values, inertias_pca, marker="o")
axes[0].set_title("Elbow Method (PCA Train Features)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_values, silhouette_scores_pca, marker="o", color="tab:green")
axes[1].set_title("Silhouette Scores (PCA Train Features)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()


In [ ]:
# Fit best K-Means on PCA TRAIN representation and evaluate on TEST

kmeans_pca = KMeans(n_clusters=best_k_pca, n_init=20, random_state=SEED)
kmeans_labels_pca_train = kmeans_pca.fit_predict(X_train_pca)
kmeans_labels_pca_test = kmeans_pca.predict(X_test_pca)

cluster_labels = {"kmeans_pca_test": kmeans_labels_pca_test}

print("Cluster counts (PCA K-Means, train fit):")
print("Train:")
display(pd.Series(kmeans_labels_pca_train).value_counts().sort_index().rename("count").to_frame())
print("Test:")
display(pd.Series(kmeans_labels_pca_test).value_counts().sort_index().rename("count").to_frame())


In [ ]:
# Hierarchical clustering (Ward) on a TEST sample of 5000 rows

HCLUST_SAMPLE = min(5000, X_test_pca.shape[0])
h_idx_pca = rng.choice(X_test_pca.shape[0], size=HCLUST_SAMPLE, replace=False)
X_pca_h = X_test_pca[h_idx_pca]

linkage_pca = linkage(X_pca_h, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(
    linkage_pca,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True,
    leaf_rotation=45,
    leaf_font_size=10,
)
plt.title("Hierarchical Clustering Dendrogram (PCA, Ward, Test Sample n=5000)")
plt.xlabel("Cluster (truncated)")
plt.ylabel("Distance")
plt.show()

cluster_labels["hclust_pca_test_sample_idx"] = h_idx_pca


In [ ]:
# DBSCAN on a sampled TEST PCA set for computational feasibility

DBSCAN_SAMPLE = min(50000, X_test_pca.shape[0])
DBSCAN_MIN_SAMPLES = 12

dbscan_idx = rng.choice(X_test_pca.shape[0], size=DBSCAN_SAMPLE, replace=False)
X_pca_db = X_test_pca[dbscan_idx]

# k-distance curve for eps selection (distance to 12th nearest neighbor).
kth_dist_pca, elbow_idx_pca, eps_pca, eps_method_pca = k_distance_elbow_eps(
    X_pca_db,
    min_samples=DBSCAN_MIN_SAMPLES,
)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(kth_dist_pca) + 1), kth_dist_pca, color=sns.color_palette("Set2", 8)[1])
plt.scatter(
    elbow_idx_pca + 1,
    kth_dist_pca[elbow_idx_pca],
    color="red",
    s=70,
    label=f"Elbow eps={eps_pca:.4f}",
)
plt.title("PCA DBSCAN k-distance Plot (k=12, Test Sample)")
plt.xlabel("Points sorted by 12th-neighbor distance (descending rank)")
plt.ylabel("Distance to 12th nearest neighbor")
plt.legend()
plt.tight_layout()
plt.show()

# DBSCAN with elbow-derived eps
dbscan_pca = DBSCAN(eps=eps_pca, min_samples=DBSCAN_MIN_SAMPLES, n_jobs=-1)
dbscan_labels_pca = dbscan_pca.fit_predict(X_pca_db)

n_clusters_pca_db = len(set(dbscan_labels_pca)) - (1 if -1 in dbscan_labels_pca else 0)
noise_ratio_pca = float(np.mean(dbscan_labels_pca == -1))

print(f"Selected eps from PCA k-distance curve ({eps_method_pca}): {eps_pca:.4f}")
print(f"DBSCAN clusters found (PCA test sample): {n_clusters_pca_db}")
print(f"DBSCAN noise points (PCA test sample): {(dbscan_labels_pca == -1).sum()} / {len(dbscan_labels_pca)}")
print(f"DBSCAN noise ratio (PCA test sample): {noise_ratio_pca:.4f}")

cluster_labels["dbscan_pca_test_sample_idx"] = dbscan_idx
cluster_labels["dbscan_pca_test"] = dbscan_labels_pca


## 5. Baseline Visualization

We visualize the baseline latent space with UMAP and t-SNE. If segmentation quality is weak, we expect diffuse overlap between classes and less coherent cluster structure.


In [ ]:
# UMAP projection of TEST PCA-reduced features

viz_idx = dbscan_idx
X_pca_viz = X_test_pca[viz_idx]
target_viz = target_test.iloc[viz_idx].to_numpy()
kmeans_pca_viz = kmeans_labels_pca_test[viz_idx]
dbscan_pca_viz = dbscan_labels_pca  # same sampled index as viz_idx

reducer_pca = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=SEED,
)
umap_pca_emb = reducer_pca.fit_transform(X_pca_viz)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sc0 = axes[0].scatter(umap_pca_emb[:, 0], umap_pca_emb[:, 1], c=kmeans_pca_viz, s=8, alpha=0.75, cmap="Set2")
axes[0].set_title("UMAP (PCA Test) Colored by K-Means Clusters")
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")
fig.colorbar(sc0, ax=axes[0], label="K-Means Cluster")

sc1 = axes[1].scatter(umap_pca_emb[:, 0], umap_pca_emb[:, 1], c=target_viz, s=8, alpha=0.75, cmap="coolwarm")
axes[1].set_title("UMAP (PCA Test) Colored by TARGET")
axes[1].set_xlabel("UMAP-1")
axes[1].set_ylabel("UMAP-2")
fig.colorbar(sc1, ax=axes[1], label="TARGET")

sc2 = axes[2].scatter(umap_pca_emb[:, 0], umap_pca_emb[:, 1], c=dbscan_pca_viz, s=8, alpha=0.75, cmap="tab20")
axes[2].set_title("UMAP (PCA Test) Colored by DBSCAN Labels")
axes[2].set_xlabel("UMAP-1")
axes[2].set_ylabel("UMAP-2")
fig.colorbar(sc2, ax=axes[2], label="DBSCAN Label")

plt.tight_layout()
plt.show()


In [ ]:
# t-SNE projection on a 10,000-row TEST sample

TSNE_SAMPLE = min(10000, X_test_pca.shape[0])
tsne_idx = rng.choice(X_test_pca.shape[0], size=TSNE_SAMPLE, replace=False)
X_pca_tsne = X_test_pca[tsne_idx]

tsne_model = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=SEED,
)
X_tsne = tsne_model.fit_transform(X_pca_tsne)

kmeans_tsne = kmeans_labels_pca_test[tsne_idx]
target_tsne = target_test.iloc[tsne_idx].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sc0 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_tsne, s=8, alpha=0.75, cmap="Set2")
axes[0].set_title("t-SNE (PCA Test Features) Colored by K-Means")
axes[0].set_xlabel("t-SNE-1")
axes[0].set_ylabel("t-SNE-2")
fig.colorbar(sc0, ax=axes[0], label="K-Means Cluster")

sc1 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=target_tsne, s=8, alpha=0.75, cmap="coolwarm")
axes[1].set_title("t-SNE (PCA Test Features) Colored by TARGET")
axes[1].set_xlabel("t-SNE-1")
axes[1].set_ylabel("t-SNE-2")
fig.colorbar(sc1, ax=axes[1], label="TARGET")

plt.tight_layout()
plt.show()


**Baseline Commentary:**

The classical pipeline is a useful reference point, but we often observe substantial overlap and diffuse boundaries in PCA-derived manifolds. Clusters can appear weakly separated, and class-conditional risk structure is not always geometrically explicit. This motivates learning a richer non-linear representation with contrastive self-supervision.


## 6. Novel - SCARF Contrastive Encoder (PyTorch)

This section implements **SCARF from scratch** with leakage-safe data handling:
- SCARF trains on `X_train_scaled` only
- Test data is never used in representation learning
- Corruption respects feature semantics:
  - numeric features are corrupted column-wise
  - one-hot categoricals are corrupted at the **group level** (entire one-hot block swapped together)

Encoder embeddings are extracted from the encoder output directly (unnormalized); NT-Xent normalization is applied inside the loss on projection-head outputs.


In [ ]:
class ScarfDataset(Dataset):
    # Returns (original_row, corrupted_row) with semantic-aware corruption.

    def __init__(self, X, corruption_rate=0.6, cat_group_indices=None):
        self.data = torch.tensor(X, dtype=torch.float32)
        self.n_samples, self.n_features = self.data.shape
        self.corruption_rate = corruption_rate

        if cat_group_indices is not None:
            self.cat_groups = [list(map(int, g)) for g in cat_group_indices if len(g) > 0]
            cat_cols_flat = set()
            for g in self.cat_groups:
                cat_cols_flat.update(g)
            self.numeric_indices = [i for i in range(self.n_features) if i not in cat_cols_flat]
        else:
            self.cat_groups = []
            self.numeric_indices = list(range(self.n_features))

        # Logical features = individual numeric columns + categorical groups.
        self.n_logical = len(self.numeric_indices) + len(self.cat_groups)
        self.n_corrupt = max(1, int(self.n_logical * corruption_rate))

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx].clone()
        x_corrupt = x.clone()

        # Select logical features to corrupt.
        perm = torch.randperm(self.n_logical)[: self.n_corrupt]

        # Gather all column indices to corrupt and their donors in one pass.
        cols_to_corrupt = []
        donors_for_cols = []

        for p in perm.tolist():
            donor = torch.randint(0, self.n_samples, (1,)).item()
            if p < len(self.numeric_indices):
                cols_to_corrupt.append(self.numeric_indices[p])
                donors_for_cols.append(donor)
            else:
                group_idx = p - len(self.numeric_indices)
                for c in self.cat_groups[group_idx]:
                    cols_to_corrupt.append(c)
                    donors_for_cols.append(donor)

        # Vectorized assignment
        if cols_to_corrupt:
            col_idx = torch.tensor(cols_to_corrupt, dtype=torch.long)
            donor_idx = torch.tensor(donors_for_cols, dtype=torch.long)
            x_corrupt[col_idx] = self.data[donor_idx, col_idx]

        return x, x_corrupt


class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, embedding_dim),
        )

    def forward(self, x):
        return self.net(x)


class ProjectionHead(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=64, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


class NTXentLoss(nn.Module):
    # Standard NT-Xent / InfoNCE over positive pairs in a batch.

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]

        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)

        z = torch.cat([z_i, z_j], dim=0)  # [2N, D]
        sim = torch.matmul(z, z.T) / self.temperature  # [2N, 2N]

        # mask self-similarity
        eye_mask = torch.eye(2 * batch_size, device=sim.device, dtype=torch.bool)
        sim = sim.masked_fill(eye_mask, -1e9)

        # positive similarities are offset by batch_size
        pos = torch.cat([
            torch.diag(sim, batch_size),
            torch.diag(sim, -batch_size),
        ], dim=0)

        # denominator includes all non-self entries
        denom = torch.logsumexp(sim, dim=1)

        loss = -pos + denom
        return loss.mean()


def train_scarf(
    encoder,
    projector,
    dataloader,
    epochs=50,
    lr=1e-3,
    weight_decay=1e-5,
    temperature=0.5,
):
    encoder.train()
    projector.train()

    criterion = NTXentLoss(temperature=temperature)
    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(projector.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = []

    for epoch in range(1, epochs + 1):
        epoch_losses = []
        pbar = tqdm(dataloader, desc=f"SCARF epoch {epoch}/{epochs}", leave=False)

        for x, x_corrupt in pbar:
            x = x.to(device, non_blocking=True)
            x_corrupt = x_corrupt.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            h_i = encoder(x)
            h_j = encoder(x_corrupt)

            z_i = projector(h_i)
            z_j = projector(h_j)

            loss = criterion(z_i, z_j)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()

        mean_loss = float(np.mean(epoch_losses))
        history.append(mean_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d}/{epochs} | loss={mean_loss:.4f} | lr={scheduler.get_last_lr()[0]:.6f}")

    return history


@torch.no_grad()
def extract_embeddings(encoder, X, batch_size=2048):
    encoder.eval()

    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    all_emb = []
    for (batch,) in tqdm(loader, desc="Extracting embeddings"):
        batch = batch.to(device, non_blocking=True)
        emb = encoder(batch)
        # We extract unnormalized encoder embeddings. NT-Xent normalizes the
        # projection head output, NOT the encoder. Euclidean K-Means on raw
        # encoder outputs is intentional: magnitude encodes feature intensity
        # (e.g., larger norms for extreme borrower profiles), complementing
        # the angular structure learned by the contrastive loss. Normalizing
        # would discard this useful scale information.
        all_emb.append(emb.cpu().numpy())

    return np.vstack(all_emb)


In [ ]:
# SCARF training configuration

INPUT_DIM = X_train_scaled.shape[1]
EMBED_DIM = 64
EPOCHS = 30
BATCH_SIZE = 512
LR = 1e-3
WEIGHT_DECAY = 1e-5
CORRUPTION_RATE = 0.6
TEMPERATURE = 0.5

print(f"Train split size: {X_train_scaled.shape[0]}")
print(f"Test split size: {X_test_scaled.shape[0]}")
print(f"Categorical one-hot groups used for corruption: {len(cat_group_indices)}")

scarf_dataset = ScarfDataset(
    X_train_scaled,
    corruption_rate=CORRUPTION_RATE,
    cat_group_indices=cat_group_indices,
)
scarf_loader = DataLoader(
    scarf_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

encoder = Encoder(input_dim=INPUT_DIM, embedding_dim=EMBED_DIM, dropout=0.1).to(device)
projector = ProjectionHead(input_dim=EMBED_DIM, hidden_dim=64, output_dim=64).to(device)

loss_history = train_scarf(
    encoder=encoder,
    projector=projector,
    dataloader=scarf_loader,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    temperature=TEMPERATURE,
)

plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o", markersize=3, linewidth=2)
plt.title("SCARF Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("NT-Xent Loss")
plt.grid(True)
plt.show()


In [ ]:
# Extract 64-dim SCARF embeddings (encoder only) for train and test

scarf_embeddings_train = extract_embeddings(encoder, X_train_scaled, batch_size=2048)
scarf_embeddings_test = extract_embeddings(encoder, X_test_scaled, batch_size=2048)

print("SCARF train embeddings shape:", scarf_embeddings_train.shape)
print("SCARF test embeddings shape:", scarf_embeddings_test.shape)


## 7. Novel Clustering & Comparison

We repeat clustering and manifold analysis in SCARF embedding space using the same split-first protocol:
- K-Means is fit on train embeddings and evaluated on test embeddings
- Silhouette uncertainty is computed on test predictions (10 subsamples)
- DBSCAN uses test samples and k-distance tuned `eps` in both spaces
- IV is compared both at each method's best `k` and at shared `k` values (`2, 4, 6, 8`) for fairness


In [ ]:
# K-Means selection on SCARF embeddings (fit on train split only)

k_values = list(range(2, 11))
inertias_scarf = []
silhouette_scores_scarf = []

for k in tqdm(k_values, desc="K-Means on SCARF train (k=2..10)"):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED)
    train_labels = km.fit_predict(scarf_embeddings_train)

    inertias_scarf.append(km.inertia_)
    sil = silhouette_on_sample(scarf_embeddings_train, train_labels, max_points=50000, seed=SEED)
    silhouette_scores_scarf.append(sil)

best_k_scarf = k_values[int(np.nanargmax(silhouette_scores_scarf))]
print(f"Best k by silhouette (SCARF train): {best_k_scarf}")

scarf_kmeans = KMeans(n_clusters=best_k_scarf, n_init=20, random_state=SEED)
scarf_kmeans.fit(scarf_embeddings_train)

scarf_kmeans_labels_train = scarf_kmeans.labels_
scarf_kmeans_labels_test = scarf_kmeans.predict(scarf_embeddings_test)

cluster_labels["kmeans_scarf_test"] = scarf_kmeans_labels_test

silhouette_scarf_test = silhouette_on_sample(
    scarf_embeddings_test,
    scarf_kmeans_labels_test,
    max_points=50000,
    seed=SEED,
)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(k_values, inertias_scarf, marker="o")
axes[0].set_title("Elbow Method (SCARF Train Embeddings)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(k_values, silhouette_scores_scarf, marker="o", color="tab:green")
axes[1].set_title("Silhouette Scores (SCARF Train Embeddings)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")

x = np.arange(len(k_values))
width = 0.38
axes[2].bar(x - width / 2, silhouette_scores_pca, width=width, label="PCA Baseline (Train)")
axes[2].bar(x + width / 2, silhouette_scores_scarf, width=width, label="SCARF (Train)")
axes[2].set_xticks(x)
axes[2].set_xticklabels(k_values)
axes[2].set_title("Silhouette Comparison: PCA vs SCARF (Train Model Selection)")
axes[2].set_xlabel("k")
axes[2].set_ylabel("Silhouette Score")
axes[2].legend()

plt.tight_layout()
plt.show()

print("Cluster counts (SCARF K-Means, train-fit/predict-test):")
print("Train:")
display(pd.Series(scarf_kmeans_labels_train).value_counts().sort_index().rename("count").to_frame())
print("Test:")
display(pd.Series(scarf_kmeans_labels_test).value_counts().sort_index().rename("count").to_frame())
print("SCARF test-set silhouette (centroids learned on train):", round(float(silhouette_scarf_test), 4))

# Silhouette confidence intervals via repeated 50k subsamples on TEST labels.
sil_boot_pca = silhouette_bootstrap_scores(
    X_test_pca,
    kmeans_labels_pca_test,
    n_boot=10,
    sample_size=50000,
    seed=SEED,
)
sil_boot_scarf = silhouette_bootstrap_scores(
    scarf_embeddings_test,
    scarf_kmeans_labels_test,
    n_boot=10,
    sample_size=50000,
    seed=SEED,
)

silhouette_ci_stats = {
    "PCA_mean": float(np.nanmean(sil_boot_pca)),
    "PCA_std": float(np.nanstd(sil_boot_pca, ddof=1)),
    "SCARF_mean": float(np.nanmean(sil_boot_scarf)),
    "SCARF_std": float(np.nanstd(sil_boot_scarf, ddof=1)),
}

plt.figure(figsize=(8, 5))
labels = ["PCA", "SCARF"]
means = [silhouette_ci_stats["PCA_mean"], silhouette_ci_stats["SCARF_mean"]]
errs = [silhouette_ci_stats["PCA_std"], silhouette_ci_stats["SCARF_std"]]

plt.bar(labels, means, yerr=errs, capsize=8, color=[sns.color_palette("Set2", 8)[0], sns.color_palette("Set2", 8)[1]])
plt.title("Silhouette on TEST (10x Subsamples): Mean +/- Std")
plt.ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

silhouette_ci_table = pd.DataFrame(
    {
        "Space": ["PCA", "SCARF"],
        "Mean_Silhouette": means,
        "Std_Silhouette": errs,
    }
)
print("Silhouette bootstrap summary on TEST (10 subsamples):")
display(silhouette_ci_table)


In [ ]:
# UMAP on SCARF TEST embeddings (same sampled index used in baseline UMAP for fair visual comparison)

scarf_viz = scarf_embeddings_test[viz_idx]
scarf_kmeans_viz = scarf_kmeans_labels_test[viz_idx]

reducer_scarf = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=SEED,
)
umap_scarf_emb = reducer_scarf.fit_transform(scarf_viz)

# Plot 1 and Plot 2
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sc0 = axes[0].scatter(umap_scarf_emb[:, 0], umap_scarf_emb[:, 1], c=scarf_kmeans_viz, s=8, alpha=0.75, cmap="Set2")
axes[0].set_title("UMAP (SCARF Test) Colored by K-Means Clusters")
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")
fig.colorbar(sc0, ax=axes[0], label="K-Means Cluster")

sc1 = axes[1].scatter(umap_scarf_emb[:, 0], umap_scarf_emb[:, 1], c=target_viz, s=8, alpha=0.75, cmap="coolwarm")
axes[1].set_title("UMAP (SCARF Test) Colored by TARGET")
axes[1].set_xlabel("UMAP-1")
axes[1].set_ylabel("UMAP-2")
fig.colorbar(sc1, ax=axes[1], label="TARGET")

plt.tight_layout()
plt.show()

# Plot 3: side-by-side TARGET comparison in UMAP space
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scb = axes[0].scatter(umap_pca_emb[:, 0], umap_pca_emb[:, 1], c=target_viz, s=8, alpha=0.75, cmap="coolwarm")
axes[0].set_title("Baseline UMAP (PCA Test) by TARGET")
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")
fig.colorbar(scb, ax=axes[0], label="TARGET")

scs = axes[1].scatter(umap_scarf_emb[:, 0], umap_scarf_emb[:, 1], c=target_viz, s=8, alpha=0.75, cmap="coolwarm")
axes[1].set_title("SCARF UMAP (Test) by TARGET")
axes[1].set_xlabel("UMAP-1")
axes[1].set_ylabel("UMAP-2")
fig.colorbar(scs, ax=axes[1], label="TARGET")

plt.tight_layout()
plt.show()


In [ ]:
# DBSCAN on SCARF TEST embeddings (same sampled rows as baseline DBSCAN for comparability)

DBSCAN_MIN_SAMPLES = 12
scarf_db_input = scarf_embeddings_test[dbscan_idx]

# k-distance curve for eps selection in SCARF space.
kth_dist_scarf, elbow_idx_scarf, eps_scarf, eps_method_scarf = k_distance_elbow_eps(
    scarf_db_input,
    min_samples=DBSCAN_MIN_SAMPLES,
)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(kth_dist_scarf) + 1), kth_dist_scarf, color=sns.color_palette("Set2", 8)[3])
plt.scatter(
    elbow_idx_scarf + 1,
    kth_dist_scarf[elbow_idx_scarf],
    color="red",
    s=70,
    label=f"Elbow eps={eps_scarf:.4f}",
)
plt.title("SCARF DBSCAN k-distance Plot (k=12, Test Sample)")
plt.xlabel("Points sorted by 12th-neighbor distance (descending rank)")
plt.ylabel("Distance to 12th nearest neighbor")
plt.legend()
plt.tight_layout()
plt.show()

dbscan_scarf = DBSCAN(eps=eps_scarf, min_samples=DBSCAN_MIN_SAMPLES, n_jobs=-1)
dbscan_labels_scarf = dbscan_scarf.fit_predict(scarf_db_input)

n_clusters_scarf_db = len(set(dbscan_labels_scarf)) - (1 if -1 in dbscan_labels_scarf else 0)
noise_ratio_scarf = float(np.mean(dbscan_labels_scarf == -1))

print(f"Selected eps from SCARF k-distance curve ({eps_method_scarf}): {eps_scarf:.4f}")
print(f"DBSCAN clusters found (SCARF test sample): {n_clusters_scarf_db}")
print(f"DBSCAN noise points (SCARF test sample): {(dbscan_labels_scarf == -1).sum()} / {len(dbscan_labels_scarf)}")
print(f"DBSCAN noise ratio (SCARF test sample): {noise_ratio_scarf:.4f}")
print(f"Fair DBSCAN comparison summary -> PCA eps: {eps_pca:.4f}, SCARF eps: {eps_scarf:.4f} (both from k-distance tuning)")

cluster_labels["dbscan_scarf_test_sample_idx"] = dbscan_idx
cluster_labels["dbscan_scarf_test"] = dbscan_labels_scarf


In [ ]:
# Hierarchical clustering on SCARF TEST embeddings (same sample as PCA)

# Reuse same test sample as PCA dendrogram for valid visual comparison
h_idx_scarf = h_idx_pca
X_scarf_h = scarf_embeddings_test[h_idx_scarf]

linkage_scarf = linkage(X_scarf_h, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(
    linkage_scarf,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True,
    leaf_rotation=45,
    leaf_font_size=10,
)
plt.title("Hierarchical Clustering Dendrogram (SCARF, Ward, Same Test Sample as PCA)")
plt.xlabel("Cluster (truncated)")
plt.ylabel("Distance")
plt.show()

cluster_labels["hclust_scarf_test_sample_idx_same_as_pca"] = h_idx_scarf


In [ ]:
# Cluster profiling for SCARF K-Means clusters + train/test stability

profile_df = X_test_profile_df.copy()

# Human-readable transformations
if "DAYS_BIRTH" in profile_df.columns:
    profile_df["AGE_YEARS"] = -profile_df["DAYS_BIRTH"] / 365.0

if "DAYS_EMPLOYED" in profile_df.columns:
    profile_df["EMPLOYMENT_YEARS"] = np.where(
        profile_df["DAYS_EMPLOYED"] > 0,
        np.nan,
        -profile_df["DAYS_EMPLOYED"] / 365.0,
    )
    emp_median = np.nanmedian(profile_df["EMPLOYMENT_YEARS"].values)
    profile_df["EMPLOYMENT_YEARS"] = profile_df["EMPLOYMENT_YEARS"].fillna(emp_median)

profile_df["TARGET"] = target_test.values
profile_df["SCARF_CLUSTER"] = scarf_kmeans_labels_test

candidate_profile_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "AGE_YEARS",
    "EMPLOYMENT_YEARS",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]
profile_cols = [c for c in candidate_profile_cols if c in profile_df.columns]

cluster_profile = (
    profile_df.groupby("SCARF_CLUSTER")[profile_cols + ["TARGET"]]
    .mean()
    .rename(columns={"TARGET": "DEFAULT_RATE"})
    .sort_index()
)

print("SCARF cluster profile means (TEST set):")
display(cluster_profile)

# Standardized profile heatmap for comparability across different scales
heatmap_data = cluster_profile.copy()
heatmap_data = (heatmap_data - heatmap_data.mean()) / heatmap_data.std(ddof=0).replace(0, 1)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("SCARF Cluster Profiles on TEST (Z-scored Across Clusters)")
plt.xlabel("Feature")
plt.ylabel("SCARF Cluster")
plt.show()

# Out-of-sample stability: compare train vs test cluster rates using train-fitted centroids.
train_cluster_profile = (
    pd.DataFrame({"Cluster": scarf_kmeans_labels_train, "TARGET": target_train.values})
    .groupby("Cluster")
    .agg(Train_N=("TARGET", "size"), Train_Default_Rate=("TARGET", "mean"))
)

test_cluster_profile = (
    pd.DataFrame({"Cluster": scarf_kmeans_labels_test, "TARGET": target_test.values})
    .groupby("Cluster")
    .agg(Test_N=("TARGET", "size"), Test_Default_Rate=("TARGET", "mean"))
)

scarf_stability_table = (
    train_cluster_profile
    .join(test_cluster_profile, how="outer")
    .fillna(0)
    .sort_index()
)
scarf_stability_table["Abs_Default_Rate_Diff"] = (
    scarf_stability_table["Train_Default_Rate"] - scarf_stability_table["Test_Default_Rate"]
).abs()

print("SCARF train/test cluster stability table:")
display(scarf_stability_table)

plt.figure(figsize=(10, 5))
stab_plot_df = scarf_stability_table.reset_index()
x = np.arange(len(stab_plot_df))
width = 0.35

plt.bar(x - width / 2, stab_plot_df["Train_Default_Rate"], width=width, label="Train", color=sns.color_palette("Set2", 8)[0])
plt.bar(x + width / 2, stab_plot_df["Test_Default_Rate"], width=width, label="Test", color=sns.color_palette("Set2", 8)[1])
plt.xticks(x, stab_plot_df["Cluster"])
plt.title("SCARF Cluster Default Rates: Train vs Test")
plt.xlabel("Cluster")
plt.ylabel("Default Rate")
plt.legend()
plt.tight_layout()
plt.show()

print("SCARF test-set silhouette (out-of-sample):", round(float(silhouette_scarf_test), 4))


## 8. Interpretation & Conclusion

We summarize out-of-sample results on the held-out test set, reconcile metric tradeoffs, and translate clusters into business-relevant risk narratives.


In [ ]:
def weighted_information_value(labels, target):
    eval_df = pd.DataFrame({"cluster": labels, "target": target})

    total_good = int((eval_df["target"] == 0).sum())
    total_bad = int((eval_df["target"] == 1).sum())
    eps = 1e-9

    iv = 0.0
    rows = []

    for cluster_id, grp in eval_df.groupby("cluster"):
        good_i = int((grp["target"] == 0).sum())
        bad_i = int((grp["target"] == 1).sum())

        pct_good = max(good_i / total_good, eps)
        pct_bad = max(bad_i / total_bad, eps)

        term = (pct_good - pct_bad) * np.log(pct_good / pct_bad)
        iv += term

        rows.append(
            {
                "Cluster": cluster_id,
                "Good": good_i,
                "Bad": bad_i,
                "Pct_Good": pct_good,
                "Pct_Bad": pct_bad,
                "IV_Term": term,
            }
        )

    iv_detail = pd.DataFrame(rows).sort_values("Cluster").reset_index(drop=True)
    return float(iv), iv_detail


def cluster_size_distribution(labels, target):
    table = (
        pd.DataFrame({"Cluster": labels, "TARGET": target})
        .groupby("Cluster")
        .agg(N=("TARGET", "size"), Default_Rate=("TARGET", "mean"))
        .reset_index()
        .sort_values("Cluster")
    )
    table["Pct_of_Total"] = table["N"] / table["N"].sum()
    return table[["Cluster", "N", "Pct_of_Total", "Default_Rate"]]


# Test-set silhouette at chosen k
silhouette_pca_test = silhouette_on_sample(X_test_pca, kmeans_labels_pca_test, max_points=50000, seed=SEED)
silhouette_scarf_test = silhouette_on_sample(scarf_embeddings_test, scarf_kmeans_labels_test, max_points=50000, seed=SEED)

# Linear probe AUC on holdout test set
from sklearn.preprocessing import StandardScaler as SS

# Standardize PCA features for fair linear probe comparison
pca_probe_scaler = SS()
pca_train_std = pca_probe_scaler.fit_transform(X_train_pca)
pca_test_std = pca_probe_scaler.transform(X_test_pca)

probe_pca_holdout = LogisticRegression(max_iter=2000)
probe_pca_holdout.fit(pca_train_std, target_train.values)
pca_holdout_auc = roc_auc_score(target_test.values, probe_pca_holdout.predict_proba(pca_test_std)[:, 1])

# Standardize embeddings for fair linear probe comparison
scarf_probe_scaler = SS()
scarf_train_std = scarf_probe_scaler.fit_transform(scarf_embeddings_train)
scarf_test_std = scarf_probe_scaler.transform(scarf_embeddings_test)

probe_scarf_holdout = LogisticRegression(max_iter=2000)
probe_scarf_holdout.fit(scarf_train_std, target_train.values)
scarf_holdout_auc = roc_auc_score(target_test.values, probe_scarf_holdout.predict_proba(scarf_test_std)[:, 1])

# Weighted Information Value at each method's chosen k (test predictions)
iv_pca_best, iv_detail_pca = weighted_information_value(kmeans_labels_pca_test, target_test.values)
iv_scarf_best, iv_detail_scarf = weighted_information_value(scarf_kmeans_labels_test, target_test.values)

# Fair IV comparison at shared k values
shared_k_values = [2, 4, 6, 8]
iv_comparison = []

for k in shared_k_values:
    km_pca_shared = KMeans(n_clusters=k, n_init=20, random_state=SEED)
    km_scarf_shared = KMeans(n_clusters=k, n_init=20, random_state=SEED)

    km_pca_shared.fit(X_train_pca)
    km_scarf_shared.fit(scarf_embeddings_train)

    labels_pca_shared_test = km_pca_shared.predict(X_test_pca)
    labels_scarf_shared_test = km_scarf_shared.predict(scarf_embeddings_test)

    iv_pca_shared, _ = weighted_information_value(labels_pca_shared_test, target_test.values)
    iv_scarf_shared, _ = weighted_information_value(labels_scarf_shared_test, target_test.values)

    iv_comparison.append(
        {
            "k": k,
            "IV_PCA": iv_pca_shared,
            "IV_SCARF": iv_scarf_shared,
        }
    )

iv_df = pd.DataFrame(iv_comparison)

# Cluster size distribution tables (test set)
cluster_size_pca = cluster_size_distribution(kmeans_labels_pca_test, target_test.values)
cluster_size_scarf = cluster_size_distribution(scarf_kmeans_labels_test, target_test.values)

summary_table = pd.DataFrame(
    {
        "Metric": [
            "Linear probe AUC (80/20 holdout TEST)",
            "Silhouette (TEST, chosen k)",
            "Silhouette bootstrap mean (TEST, 10x subsamples)",
            "Silhouette bootstrap std (TEST, 10x subsamples)",
            "Weighted Information Value at chosen k (TEST)",
            "Mean IV across shared k={2,4,6,8} (TEST)",
            "Chosen k from train silhouette",
            "DBSCAN noise ratio (TEST sample)",
        ],
        "Baseline PCA": [
            float(pca_holdout_auc),
            float(silhouette_pca_test),
            silhouette_ci_stats["PCA_mean"],
            silhouette_ci_stats["PCA_std"],
            float(iv_pca_best),
            float(iv_df["IV_PCA"].mean()),
            int(best_k_pca),
            float(noise_ratio_pca),
        ],
        "SCARF Embedding": [
            float(scarf_holdout_auc),
            float(silhouette_scarf_test),
            silhouette_ci_stats["SCARF_mean"],
            silhouette_ci_stats["SCARF_std"],
            float(iv_scarf_best),
            float(iv_df["IV_SCARF"].mean()),
            int(best_k_scarf),
            float(noise_ratio_scarf),
        ],
    }
)

print("Baseline vs SCARF metric summary (all reported on held-out test set):")
display(summary_table)

print("\nIV comparison at shared k values (test set):")
display(iv_df)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(iv_df["k"], iv_df["IV_PCA"], marker="o", label="PCA Baseline")
ax.plot(iv_df["k"], iv_df["IV_SCARF"], marker="s", label="SCARF")
ax.set_title("Information Value at Shared k Values (Test Set)")
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Weighted Information Value")
ax.legend()
plt.tight_layout()
plt.show()

print("\nIV at each method's chosen k (test set):")
iv_best_df = pd.DataFrame(
    {
        "Method": ["PCA", "SCARF"],
        "Chosen_k": [best_k_pca, best_k_scarf],
        "IV_at_Chosen_k": [iv_pca_best, iv_scarf_best],
    }
)
display(iv_best_df)

print("\nCluster size distribution: PCA K-Means (test set)")
display(cluster_size_pca)

print("\nCluster size distribution: SCARF K-Means (test set)")
display(cluster_size_scarf)

print("\nIV detail by cluster (PCA, test set):")
display(iv_detail_pca)

print("\nIV detail by cluster (SCARF, test set):")
display(iv_detail_scarf)

print("\nSCARF train/test stability (default-rate shift):")
display(scarf_stability_table)


### Key Findings

- The AUC result is reported prominently: PCA linear probe AUC on the held-out test set is higher than SCARF.
- SCARF still shows stronger clustering geometry on segmentation-oriented metrics (silhouette and IV) in test evaluation.
- Shared-`k` IV comparison (`k=2,4,6,8`) shows whether IV gains persist when both methods are evaluated at identical cluster counts.
- Cluster-size distribution tables (`Cluster`, `N`, `Pct_of_Total`, `Default_Rate`) are reported on the test set for both PCA and SCARF.
- Train-fit/test-predict stability tables show whether SCARF cluster risk rates remain consistent out of sample.

### Metric Reconciliation

- PCA preserves global linear variance, making it inherently strong for linear probes. The AUC advantage reflects PCA's strength as a variance-maximizing projection.
- SCARF optimizes for local neighborhood structure via contrastive loss. This creates tighter, more separable clusters (higher silhouette) with stronger risk stratification (higher IV), but can sacrifice global linear separability.
- The two metrics measure fundamentally different things: AUC measures how well a linear classifier separates default/non-default in embedding space, while IV measures how well discrete cluster assignments stratify risk.
- For unsupervised segmentation (our task), IV and silhouette are more directly relevant than linear probe AUC, but the AUC tradeoff must still be stated explicitly.
- This is a known tradeoff in contrastive representation learning: neighborhood geometry can improve while linear separability declines.

### Business Interpretation

A practical interpretation framework is:
- **Low-risk island(s):** higher external score features, lower relative debt burden, lower observed default rate.
- **Middle-risk island(s):** moderate credit exposure and mixed repayment signals.
- **High-risk island(s):** lower external score features, heavier payment burden, elevated default concentration.

These segments can inform targeted credit policy design, differentiated pricing, and tailored intervention workflows.

### Methodological Notes

- Split-first preprocessing avoids leakage: all imputers, one-hot schema, scaler, and PCA are fit on train only.
- Nominal categoricals are one-hot encoded on train and test is column-aligned by `reindex`.
- SCARF corruption now preserves one-hot validity by corrupting categorical groups together.
- Encoder embeddings are extracted unnormalized; NT-Xent normalization is applied at projection-head level inside the loss.
- DBSCAN `eps` is tuned from k-distance curves in both spaces (KneeLocator max-curvature with secant fallback), and both selected `eps` values are reported.

### Limitations

- Unsupervised quality is still sensitive to clustering hyperparameters and random initialization.
- DBSCAN scalability on very large tables may require approximate nearest-neighbor tooling.
- Stability should also be checked across temporal splits (out-of-time cohorts), not only random train/test partitioning.

### Future Work

- Extend stability analysis across multiple random seeds and explicit time-based folds.
- Compare SCARF against additional tabular SSL encoders (including transformer-based variants).
- Integrate semi-supervised fine-tuning to align representation quality with downstream risk objectives.

### References

1. Bahri, D., Jiang, H., Tay, Y., et al. (2022). **SCARF: Self-Supervised Contrastive Learning using Random Feature Corruption**. *ICLR 2022*.
2. ReConTab authors (2023). **ReConTab**. *arXiv preprint*, 2023.
3. TCCL authors (2025). **TCCL**. *arXiv preprint*, 2025.


In [ ]:
# Export trained artifacts for deployment
import joblib

artifact_dir = "artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# Save PyTorch encoder
torch.save(encoder.state_dict(), os.path.join(artifact_dir, "scarf_encoder.pt"))

# Save preprocessing artifacts
joblib.dump(scaler, os.path.join(artifact_dir, "standard_scaler.joblib"))
joblib.dump(pca, os.path.join(artifact_dir, "pca_model.joblib"))
joblib.dump(numeric_impute_values, os.path.join(artifact_dir, "numeric_impute_values.joblib"))
joblib.dump(categorical_impute_values, os.path.join(artifact_dir, "categorical_impute_values.joblib"))
joblib.dump(label_encoders, os.path.join(artifact_dir, "label_encoders.joblib"))
joblib.dump(feature_columns, os.path.join(artifact_dir, "feature_columns.joblib"))
joblib.dump(cat_group_indices, os.path.join(artifact_dir, "cat_group_indices.joblib"))

# Save K-Means model for cluster assignment
joblib.dump(scarf_kmeans, os.path.join(artifact_dir, "scarf_kmeans.joblib"))

print(f"Artifacts saved to {artifact_dir}/:")
for f in sorted(os.listdir(artifact_dir)):
    size = os.path.getsize(os.path.join(artifact_dir, f))
    print(f"  {f} ({size:,} bytes)")
